In [80]:
import pandas as pd


In [81]:
df = pd.read_csv("/content/100_Unique_QA_Dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [82]:
df.shape

(90, 2)

In [83]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace("?","")
  text = text.replace("'","")
  return text.split()


In [84]:
tokenize("What is the boiling point of water in Celsius?	")

['what', 'is', 'the', 'boiling', 'point', 'of', 'water', 'in', 'celsius']

In [85]:
# vocabulary
vocab = {'UNK':0}

In [86]:
def build_vocab(row):

  tokenize_question = tokenize(row['question'])
  tokenize_answer = tokenize(row['answer'])\

  merged_tokens = tokenize_question + tokenize_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab)

In [87]:
df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [88]:
len(vocab)

324

In [89]:
from IPython.core.inputtransformer2 import TokenTransformBase
# convert words to numerical indices
def text_to_indices(text,vocab):
  indexed_text = []

  for tokens  in tokenize(text):

    if tokens in vocab:
      indexed_text.append(vocab[tokens])
    else:
      indexed_text.append(vocab['UNK'])

  return indexed_text

In [90]:
text_to_indices("Where is Nepal",vocab)

[0, 2, 0]

In [91]:
import torch
from torch.utils.data import Dataset,DataLoader

In [92]:
class QADataset(Dataset):

  def __init__(self,df,vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]


  def __getitem__(self, index):
    self.df.iloc[index]['question']
    self.df.iloc[index]['answer']

    numerical_question = text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'],self.vocab)

    return torch.tensor (numerical_question),torch.tensor(numerical_answer)

In [93]:
dataset = QADataset(df,vocab)

In [94]:
dataloader = DataLoader(dataset,batch_size=1,shuffle=True)

In [95]:
for question ,answer in dataset:
  print(question,answer)

tensor([1, 2, 3, 4, 5, 6]) tensor([7])
tensor([1, 2, 3, 4, 5, 8]) tensor([9])
tensor([10, 11, 12, 13, 14, 15]) tensor([16])
tensor([ 1,  2,  3, 17, 18, 19, 20, 21, 22]) tensor([23])
tensor([ 1,  2,  3, 24, 25,  5, 26, 19, 27]) tensor([28])
tensor([10, 29,  3, 30, 31]) tensor([32])
tensor([ 1,  2,  3, 33, 34,  5, 35]) tensor([36])
tensor([ 1,  2,  3, 37, 38, 39, 40]) tensor([41])
tensor([42, 43, 44, 45, 46, 47, 48]) tensor([49])
tensor([ 1,  2,  3, 50, 51, 19,  3, 45]) tensor([52])
tensor([ 1,  2,  3,  4,  5, 53]) tensor([54])
tensor([10, 55,  3, 56,  5, 57]) tensor([58])
tensor([ 1,  2,  3, 59, 25,  5, 26, 19, 60]) tensor([61])
tensor([42, 18,  2, 62, 63,  3, 64, 18]) tensor([65])
tensor([10,  2,  3, 66,  5, 67]) tensor([68])
tensor([ 1,  2,  3, 69,  5,  3, 70, 71]) tensor([72])
tensor([ 1,  2,  3,  4,  5, 73]) tensor([74])
tensor([10, 75, 76]) tensor([77])
tensor([78, 79, 80, 81, 82, 83, 84]) tensor([85])
tensor([42, 86, 87, 88, 89, 39, 90]) tensor([91])
tensor([ 1,  2,  3, 92, 93, 94

In [96]:
import torch.nn as nn

In [97]:
class SimpleRNN(nn.Module):

  def __init__(self,vocab_size):

    super().__init__()
    self.embedding = nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn = nn.RNN(50,64,batch_first=True)
    self.fc = nn.Linear(64,vocab_size)

  def forward(self,question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [98]:
x = nn.Embedding(324,embedding_dim=50)
y = nn.RNN(50,64,batch_first=True)
z = nn.Linear(64,324)

a = dataset[0][0].reshape(1,6)
print("Shape of a:",a.shape)
b = x(a)
print("Shape of b:",b.shape)
c,d = y(b)
print("Shape of c:",c.shape)
print("Shape of d:",d.shape)

e = z(d.squeeze(0))
print("Shape of e:",e.shape)

Shape of a: torch.Size([1, 6])
Shape of b: torch.Size([1, 6, 50])
Shape of c: torch.Size([1, 6, 64])
Shape of d: torch.Size([1, 1, 64])
Shape of e: torch.Size([1, 324])


In [99]:
learning_rate = 0.001
epochs = 20

In [100]:
model = SimpleRNN(len(vocab))

In [101]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [102]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 528.512377
Epoch: 2, Loss: 458.404738
Epoch: 3, Loss: 376.798284
Epoch: 4, Loss: 315.567320
Epoch: 5, Loss: 262.586207
Epoch: 6, Loss: 213.544209
Epoch: 7, Loss: 168.882600
Epoch: 8, Loss: 131.681652
Epoch: 9, Loss: 101.301045
Epoch: 10, Loss: 77.712478
Epoch: 11, Loss: 60.399016
Epoch: 12, Loss: 47.641773
Epoch: 13, Loss: 38.298806
Epoch: 14, Loss: 30.907938
Epoch: 15, Loss: 25.678562
Epoch: 16, Loss: 21.456343
Epoch: 17, Loss: 18.203105
Epoch: 18, Loss: 15.473189
Epoch: 19, Loss: 13.400856
Epoch: 20, Loss: 11.494609


In [103]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [104]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [105]:
list(vocab.keys())[7]

'paris'